# Python程序设计（项目实战） · 工单P2.1《Pandas数据处理管道》 — 教师参考版
第3周 | 参考实现可直接运行验证 | 答案请勿下发学生

In [1]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())


工作目录: /Users/joelgong/Desktop/现代制造 1/Python编程实战/教学实施资料/03-Python程序设计-项目实战


## 参考实现 `p21`（含依赖导入）

In [2]:
import sys
import numpy as np  # 通用科学计算


def p21():
    import pandas as pd

    df = pd.read_csv("W03_raw.csv")
    log = [f"加载 {len(df)} 行"]
    log.append(f"temp缺失 {df.temp.isna().sum()}")

    df["temp"] = pd.to_numeric(df.temp, errors="coerce")     # 类型错→NaN
    parts = []
    for fmt in ("%Y-%m-%d %H:%M:%S", "%m/%d/%Y %H:%M", "%Y%m%d%H%M%S"):
        parts.append(pd.to_datetime(df.ts, format=fmt, errors="coerce"))
    ts = parts[0]
    for p in parts[1:]:
        ts = ts.combine_first(p)                            # 三格式统一
    df["ts"] = ts

    df["temp"] = df.temp.interpolate()                       # 决策: 时序插值
    q1, q3 = df.temp.quantile([.25, .75])
    iqr = q3 - q1
    biz = df.temp > 120                                     # 业务阈值
    df = df[~((df.temp < q1 - 1.5 * iqr) | (df.temp > q3 + 1.5 * iqr) | biz)]
    df = df.drop_duplicates(subset=["device_id", "ts"])
    df.to_csv("clean.csv", index=False)

    s = (40 * (1 - df.isna().mean().mean())
         + 30 * df.temp.between(20, 120).mean()
         + 20 * df.device_id.str.match(r"DEV-\d{2}").mean()
         + 10)
    print("\n".join(log))
    print(f"清洗后 {len(df)} 行, 质量评分 {round(s, 1)}")


## 学生版TODO → 教师答案对照
学生版starter中的TODO与上方参考实现（或下方双解）的对应关系：

| TODO | 题面（学生看到的指导） |
|------|----------------------|
| `2` | TODO-2 缺失处理(决策: dropna/fillna/interpolate——写矩阵后选定) |
| `3` | TODO-3 类型转换: temp→float(错误值→NaN); ts→datetime(三种格式统一!) |
| `4` | TODO-4 异常检测: IQR法 与 业务阈值(>120) 的AND结果(决策记录理由) |
| `5` | TODO-5 去重: subset=['device_id','ts'] |
| `6` | TODO-6 验证: temp范围[20,120], press[0.1,1.0] |
| `7` | TODO-7 输出: clean.csv + 处理日志 |

> 发放纪律：本对照表**勿随学生版下发**；课堂按层放提示（D层演示照做/C层只给思路行）。

## 多解对比（已实跑互验）
| 解 | 路线 | 结论 |
|----|------|------|
| A | pandas链式清洗 | 大数据/后续聚合（3行留存与B一致） |
| B | 纯Python字典归并 | 无pandas环境/嵌入式网关 |

> 决策素材：产线分析机→A；边缘网关→B。

## 运行环境(如需)

In [3]:
import subprocess, sys
subprocess.run([sys.executable, '学生代码框架/W03_pipeline.py', '--peek'], check=True)
print('样例脏数据已生成: W03_raw.csv')

  device_id                ts    temp  press status
0    DEV-01  03/01/2026 00:00  137.60   0.45    RUN
1    DEV-02    20260301003000   79.40   0.33    RUN
2    DEV-03    20260301010000   53.93   0.48    RUN
3    DEV-04    20260301013000   67.26   0.54    RUN
4    DEV-01    20260301020000   66.91   0.52    RUN
5    DEV-02    20260301023000   75.72   0.53    RUN
6    DEV-02    20260301023000   75.72   0.53    RUN
7    DEV-03    20260301030000   68.46   0.37    RUN
8    DEV-04  03/01/2026 03:30   69.57   0.33    RUN
9    DEV-01    20260301040000   77.67   0.43    RUN
temp异常值预览:
 Empty DataFrame
Columns: [device_id, ts, temp, press, status]
Index: []
样例脏数据已生成: W03_raw.csv


## 执行与验证

In [4]:
p21()

加载 1206 行
temp缺失 55
清洗后 1186 行, 质量评分 100.0
